# 2층 신경망 역전파 유도

고정 예제는 입력 2개, 은닉 2개, 출력 1개의 완전연결 신경망이다. 은닉층과 출력층 활성화 함수는 모두 Sigmoid이고, 손실 함수는 Binary Cross-Entropy이다. 정답은 `y_true = 1`로 둔다.

| 기호 | shape | 값 |
|---|---:|---|
| x | (2,) | [1, 0] |
| W1 | (2,2) | [[0.1, 0.2], [0.3, 0.4]] |
| b1 | (2,) | [0, 0] |
| W2 | (2,) | [0.5, 0.6] |
| b2 | scalar | 0 |
| y_true | scalar | 1 |

## 순전파 수식과 손계산

Sigmoid 함수는 `sigma(t) = 1 / (1 + exp(-t))`이다.

1. `z1 = W1 @ x + b1`, shape `(2,)`
   - `z1 = [[0.1, 0.2], [0.3, 0.4]] @ [1, 0] + [0, 0] = [0.1, 0.3]`
2. `a1 = sigmoid(z1)`, shape `(2,)`
   - `a1 = [sigmoid(0.1), sigmoid(0.3)] = [0.5250, 0.5744]`
3. `z2 = W2 @ a1 + b2`, scalar
   - `z2 = 0.5 * 0.5250 + 0.6 * 0.5744 = 0.6072`
4. `y_pred = sigmoid(z2)`, scalar
   - `y_pred = sigmoid(0.6072) = 0.6473`
5. `L = -(y log(y_pred) + (1-y) log(1-y_pred))`, scalar
   - `y_true = 1`이므로 `L = -log(0.6473) = 0.4349`

## 역전파 수식 유도

출력층부터 입력층 방향으로 연쇄 법칙을 적용한다.

1. `dL/dy_pred`, scalar
   - `L = -(y log(y_pred) + (1-y) log(1-y_pred))`
   - `dL/dy_pred = -y/y_pred + (1-y)/(1-y_pred)`
   - `y=1`이면 `dL/dy_pred = -1/y_pred = -1.5449`
2. `dL/dz2`, scalar
   - `dy_pred/dz2 = y_pred(1-y_pred)`
   - BCE와 Sigmoid를 결합하면 `dL/dz2 = y_pred - y_true = -0.3527`
3. `dL/dW2`, shape `(2,)`
   - `z2 = W2 @ a1 + b2`
   - `dL/dW2 = dL/dz2 * a1 = [-0.1852, -0.2026]`
4. `dL/da1`, shape `(2,)`
   - `dL/da1 = dL/dz2 * W2 = [-0.1764, -0.2116]`
5. `dL/dz1`, shape `(2,)`
   - `da1/dz1 = a1 * (1-a1)`
   - `dL/dz1 = dL/da1 * a1 * (1-a1) = [-0.0440, -0.0517]`
6. `dL/dW1`, shape `(2,2)`
   - `z1 = W1 @ x + b1`
   - `dL/dW1 = outer(dL/dz1, x)`
   - `dL/dW1 = [[-0.0440, 0.0000], [-0.0517, 0.0000]]`

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.array([1.0, 0.0])
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([0.0, 0.0])
W2 = np.array([0.5, 0.6])
b2 = 0.0
y_true = 1.0

z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
y_pred = sigmoid(z2)
loss = -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

dL_dy_pred = -(y_true / y_pred) + (1 - y_true) / (1 - y_pred)
dL_dz2 = y_pred - y_true
dL_dW2 = dL_dz2 * a1
dL_da1 = dL_dz2 * W2
dL_dz1 = dL_da1 * a1 * (1 - a1)
dL_dW1 = np.outer(dL_dz1, x)

for name, value in {
    'z1': z1,
    'a1': a1,
    'z2': z2,
    'y_pred': y_pred,
    'loss': loss,
    'dL/dy_pred': dL_dy_pred,
    'dL/dz2': dL_dz2,
    'dL/dW2': dL_dW2,
    'dL/da1': dL_da1,
    'dL/dz1': dL_dz1,
    'dL/dW1': dL_dW1,
}.items():
    print(name, np.round(value, 4))

## NumPy 검증 결과

| 항목 | 손계산(4자리) | NumPy(4자리) | shape |
|---|---:|---:|---:|
| z1 | [0.1000, 0.3000] | [0.1000, 0.3000] | (2,) |
| a1 | [0.5250, 0.5744] | [0.5250, 0.5744] | (2,) |
| z2 | 0.6072 | 0.6072 | scalar |
| y_pred | 0.6473 | 0.6473 | scalar |
| dL/dy_pred | -1.5449 | -1.5449 | scalar |
| dL/dz2 | -0.3527 | -0.3527 | scalar |
| dL/dW2 | [-0.1852, -0.2026] | [-0.1852, -0.2026] | (2,) |
| dL/da1 | [-0.1764, -0.2116] | [-0.1764, -0.2116] | (2,) |
| dL/dz1 | [-0.0440, -0.0517] | [-0.0440, -0.0517] | (2,) |
| dL/dW1 | [[-0.0440, 0.0000], [-0.0517, 0.0000]] | [[-0.0440, 0.0000], [-0.0517, 0.0000]] | (2,2) |

모든 순전파 중간값과 역전파 기울기가 소수점 4자리까지 손계산과 일치한다.